In [15]:

from agents import Agent, Runner, function_tool, ItemHelpers

@function_tool
def get_weather(city:str):
    """Get the weather in a specific city"""
    print(city)
    return "30 degress"



agent = Agent(
    name="Assistant Agent",
    instructions="You ar a helpful Assistant. Use tools when needed to answer questions",
    tools=[get_weather]
)
 
## 스트리밍 출력
stream = Runner.run_streamed(agent, "Hello how are you? what is the weather in tokyo?")

async for event in stream.stream_events():
    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print("Agent updated to", event.new_agent.name)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict())
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item))
    print('='*20)


Agent updated to Assistant Agent
{'arguments': '{"city":"Tokyo"}', 'call_id': 'call_ZGMQH42k5TQPduU7T9d9jOrs', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_044f2c09d6d20b89006a3e8d0a2c5c819aa25361338614e338', 'status': 'completed'}
Tokyo
30 degress
MessageOutputItem(agent=Agent(name='Assistant Agent', handoff_description=None, tools=[FunctionTool(name='get_weather', description='Get the weather in a specific city', params_json_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x10d350b90>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instruct

In [ ]:
from agents import Agent, Runner, function_tool, SQLiteSession

session = SQLiteSession("user_1", "ai-memory.db")

@function_tool
def get_weather(city: str):
    """Get the weather in a specific city"""
    print(city)
    return "30 degress"

agent = Agent(
    name="Assistant Agent",
    instructions="You ar a helpful Assistant. Use tools when needed to answer questions",
    tools=[get_weather],
)

result = await Runner.run(
    agent,
    "私は日本に住んでいる。",
    session=session,
)

print(result.final_output)

In [12]:
result = await Runner.run(
    agent,
    "私が住んでいる国の中で首都の明日の天気は？",
    session=session,
)

print(result.final_output)

はい、承知しました。


In [13]:
# 履歴が空のセッションに手動で文脈を注入する例
session2 = SQLiteSession("user_2", "ai-memory.db")

await session2.add_items([{"role": "user", "content": "私は日本に住んでいる。"}])

result = await Runner.run(
    agent,
    "私が住んでいる国の中で首都の明日の天気は？",
    session=session2,
)

print(result.final_output)